# Notebook 2 — Link, guard, execute


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "text2sql.py").exists():
    raise SystemExit("Set the notebook's working directory to 17_agentic_text2sql.")
sys.path.insert(0, str(HERE))

import text2sql as desk

if not desk.ollama_alive():
    raise SystemExit("Start Ollama and pull qwen3.5:4b before the scoring cell.")
print("Ollama: up")


Ollama: up


## Which tables, and why

HarborDesk leave matches no card and no table name, so the link is empty. The desk will abstain. It will not invent a SQL query for a question this bank cannot hold.

Pisek brings `district` (the value `Pisek` lives on `A2`) and `account` (the question names that table). It does not bring `trans`.


In [2]:
samples = [
    "How many annual leave days does a full-time HarborDesk employee receive?",
    "How many accounts are there in the district of Pisek?",
    "What type of credit card does the client with id 13539 own?",
    "What is the average loan amount for male account owners?",
]
for question in samples:
    print(desk.link_tables(question), "|", question)


[] | How many annual leave days does a full-time HarborDesk employee receive?
['account', 'district'] | How many accounts are there in the district of Pisek?
['card', 'client', 'disp'] | What type of credit card does the client with id 13539 own?
['loan', 'client', 'account', 'disp'] | What is the average loan amount for male account owners?


## The guard

A read-only connection would also reject a `DELETE`. The error would look like a database failure. `guard_sql` states the rule in one sentence: one SELECT, no write, no second statement.


In [3]:
for statement in [
    "SELECT COUNT(*) FROM account",
    "DELETE FROM loan WHERE status = 'D'",
    "SELECT 1; DROP TABLE loan",
]:
    print(desk.guard_sql(statement) or "allowed", "|", statement)


allowed | SELECT COUNT(*) FROM account
This is not a read. Insert, update, delete, and drop are blocked. | DELETE FROM loan WHERE status = 'D'
More than one statement. The desk runs a single SELECT. | SELECT 1; DROP TABLE loan


## The score that matters

`sql_report` links, writes one SELECT, runs it, and repairs once if the database reports an error or a zero. A match means the cells agree with `questions.json`. The SQL text does not have to be identical.

On the seeded run, eight of the nine questions match. The miss is the unemployment question. `A12` is the 1995 rate and `A13` is the 1996 rate. The model treats those columns as dates. That is a reading bug, which is what notebook 3's person is for when the result is a number. Here the result is the wrong number, so the row is a miss.

The cell calls Qwen once or twice per question. It takes about a minute.


In [4]:
rows = desk.sql_report()
matched = sum(row["match"] for row in rows)
print(f"result match {matched}/{len(rows)}\n")
for row in rows:
    mark = "ok  " if row["match"] else "MISS"
    print(f"{mark} {row['id']:32} got={row['rows']} expected={row['expected']}")
    if not row["match"]:
        print("   ", row["sql"])


result match 8/9

ok   pisek_accounts                   got=[[46]] expected=[[46]]
ok   praha_men                        got=[[339]] expected=[[339]]
ok   card_type                        got=[['junior']] expected=[['junior']]
ok   gold_owner_cards                 got=[[88]] expected=[[88]]
ok   statement_after_transaction      got=[[93]] expected=[[93]]
ok   east_bohemia_after_transaction   got=[[13]] expected=[[13]]
MISS unemployment_year                got=[[0, 0]] expected=[['1996']]
    SELECT MAX(CASE WHEN strftime('%Y', A12) = '1995' THEN 1 ELSE 0 END) AS year_1995, MAX(CASE WHEN strftime('%Y', A13) = '1996' THEN 1 ELSE 0 END) AS year_1996 FROM district ORDER BY CAST(year_1995 AS INTEGER), CAST(year_1996 AS INTEGER) DESC LIMIT 1
ok   accounts_opened_1993             got=[[1139]] expected=[[1139]]
ok   avg_loan_male_owner              got=[[147501.7365269461]] expected=[[147501.74]]


## Exercise

Run `link_tables` on these two questions and write down the tables.

1. `How many male clients live in the district Hl.m. Praha?`
2. `How many accounts were opened in 1993?`

<details><summary>Answer</summary>

1. `client` and `district`. Gender is `M`, and the district name is the stored value `Hl.m. Praha`.
2. `account`. The year filter is `strftime('%Y', date) = '1993'`.

</details>


In [5]:
for question in [
    "How many male clients live in the district Hl.m. Praha?",
    "How many accounts were opened in 1993?",
]:
    print(desk.link_tables(question), "|", question)


['client', 'district'] | How many male clients live in the district Hl.m. Praha?
['account'] | How many accounts were opened in 1993?
